new

In [0]:
# Load Healthcare Dataset

file_path = "/Volumes/workspace/default/facility_capability_data/treatments*.csv"

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

# Fill missing values
df = df.fillna("Unknown")

# Preview dataset
display(df)

treatment_id,appointment_id,treatment_type,description,cost,treatment_date
T001,A001,Chemotherapy,Basic screening,3941.97,2023-08-09
T002,A002,MRI,Advanced protocol,4158.44,2023-06-09
T003,A003,MRI,Standard procedure,3731.55,2023-06-28
T004,A004,MRI,Basic screening,4799.86,2023-09-01
T005,A005,ECG,Standard procedure,582.05,2023-07-06
T006,A006,Chemotherapy,Standard procedure,1381.0,2023-06-19
T007,A007,Chemotherapy,Advanced protocol,534.03,2023-04-09
T008,A008,Physiotherapy,Basic screening,3413.64,2023-05-24
T009,A009,Physiotherapy,Standard procedure,4541.14,2023-03-05
T010,A010,Physiotherapy,Standard procedure,1595.67,2023-01-13


In [0]:
from pyspark.sql.functions import concat_ws

# Combine important text columns

df = df.withColumn(
    "combined_text",
    concat_ws(
        " ",
        df["treatment_type"],
        df["description"]
    )
)

display(df)

treatment_id,appointment_id,treatment_type,description,cost,treatment_date,combined_text
T001,A001,Chemotherapy,Basic screening,3941.97,2023-08-09,Chemotherapy Basic screening
T002,A002,MRI,Advanced protocol,4158.44,2023-06-09,MRI Advanced protocol
T003,A003,MRI,Standard procedure,3731.55,2023-06-28,MRI Standard procedure
T004,A004,MRI,Basic screening,4799.86,2023-09-01,MRI Basic screening
T005,A005,ECG,Standard procedure,582.05,2023-07-06,ECG Standard procedure
T006,A006,Chemotherapy,Standard procedure,1381.0,2023-06-19,Chemotherapy Standard procedure
T007,A007,Chemotherapy,Advanced protocol,534.03,2023-04-09,Chemotherapy Advanced protocol
T008,A008,Physiotherapy,Basic screening,3413.64,2023-05-24,Physiotherapy Basic screening
T009,A009,Physiotherapy,Standard procedure,4541.14,2023-03-05,Physiotherapy Standard procedure
T010,A010,Physiotherapy,Standard procedure,1595.67,2023-01-13,Physiotherapy Standard procedure


In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Function to detect healthcare capabilities

def detect_capability(text):

    if text is None:
        return "General Treatment"

    text = text.lower()

    if "surgery" in text:
        return "Surgery"

    elif "cardio" in text:
        return "Cardiology"

    elif "emergency" in text:
        return "Emergency Care"

    elif "ortho" in text:
        return "Orthopedic Care"

    elif "neuro" in text:
        return "Neurology"

    elif "dialysis" in text:
        return "Dialysis"

    elif "x-ray" in text:
        return "Radiology"

    elif "scan" in text:
        return "Imaging"

    else:
        return "General Treatment"

# Create UDF
detect_udf = udf(detect_capability, StringType())

# Apply intelligent parsing
df = df.withColumn(
    "detected_features",
    detect_udf(df["combined_text"])
)

# Preview processed data
display(df)

treatment_id,appointment_id,treatment_type,description,cost,treatment_date,combined_text,detected_features
T001,A001,Chemotherapy,Basic screening,3941.97,2023-08-09,Chemotherapy Basic screening,General Treatment
T002,A002,MRI,Advanced protocol,4158.44,2023-06-09,MRI Advanced protocol,General Treatment
T003,A003,MRI,Standard procedure,3731.55,2023-06-28,MRI Standard procedure,General Treatment
T004,A004,MRI,Basic screening,4799.86,2023-09-01,MRI Basic screening,General Treatment
T005,A005,ECG,Standard procedure,582.05,2023-07-06,ECG Standard procedure,General Treatment
T006,A006,Chemotherapy,Standard procedure,1381.0,2023-06-19,Chemotherapy Standard procedure,General Treatment
T007,A007,Chemotherapy,Advanced protocol,534.03,2023-04-09,Chemotherapy Advanced protocol,General Treatment
T008,A008,Physiotherapy,Basic screening,3413.64,2023-05-24,Physiotherapy Basic screening,General Treatment
T009,A009,Physiotherapy,Standard procedure,4541.14,2023-03-05,Physiotherapy Standard procedure,General Treatment
T010,A010,Physiotherapy,Standard procedure,1595.67,2023-01-13,Physiotherapy Standard procedure,General Treatment


In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("healthcare_table_new")

print("New Delta table created successfully")

New Delta table created successfully


In [0]:
%sql
SELECT treatment_type,
COUNT(*) AS total_cases
FROM healthcare_table_new
GROUP BY treatment_type
ORDER BY total_cases DESC;

treatment_type,total_cases
Chemotherapy,49
X-Ray,41
ECG,38
MRI,36
Physiotherapy,36


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT detected_features,
COUNT(*) AS feature_count
FROM healthcare_table_new
GROUP BY detected_features
ORDER BY feature_count DESC;

detected_features,feature_count
General Treatment,159
Radiology,41


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT detected_features,
AVG(cost) AS average_cost
FROM healthcare_table_new
GROUP BY detected_features
ORDER BY average_cost DESC;

detected_features,average_cost
General Treatment,2771.0451572327042
Radiology,2698.8700000000003


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT treatment_type,
SUM(cost) AS total_cost
FROM healthcare_table_new
GROUP BY treatment_type
ORDER BY total_cost DESC;

treatment_type,total_cost
Chemotherapy,128855.68
MRI,116098.15999999999
X-Ray,110653.67000000001
Physiotherapy,99418.1
ECG,96224.24


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT treatment_type,
cost
FROM healthcare_table_new
ORDER BY cost DESC
LIMIT 10;

treatment_type,cost
X-Ray,4973.63
MRI,4966.18
Chemotherapy,4964.71
ECG,4960.65
Chemotherapy,4945.03
X-Ray,4890.25
Physiotherapy,4846.2
Chemotherapy,4834.02
X-Ray,4833.17
X-Ray,4809.31


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT detected_features,
COUNT(*) AS total_cases
FROM healthcare_table_new
GROUP BY detected_features
HAVING COUNT(*) > 1
ORDER BY total_cases DESC;

detected_features,total_cases
General Treatment,159
Radiology,41


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT *
FROM healthcare_table_new
LIMIT 20;

treatment_id,appointment_id,treatment_type,description,cost,treatment_date,combined_text,detected_features
T001,A001,Chemotherapy,Basic screening,3941.97,2023-08-09,Chemotherapy Basic screening,General Treatment
T002,A002,MRI,Advanced protocol,4158.44,2023-06-09,MRI Advanced protocol,General Treatment
T003,A003,MRI,Standard procedure,3731.55,2023-06-28,MRI Standard procedure,General Treatment
T004,A004,MRI,Basic screening,4799.86,2023-09-01,MRI Basic screening,General Treatment
T005,A005,ECG,Standard procedure,582.05,2023-07-06,ECG Standard procedure,General Treatment
T006,A006,Chemotherapy,Standard procedure,1381.0,2023-06-19,Chemotherapy Standard procedure,General Treatment
T007,A007,Chemotherapy,Advanced protocol,534.03,2023-04-09,Chemotherapy Advanced protocol,General Treatment
T008,A008,Physiotherapy,Basic screening,3413.64,2023-05-24,Physiotherapy Basic screening,General Treatment
T009,A009,Physiotherapy,Standard procedure,4541.14,2023-03-05,Physiotherapy Standard procedure,General Treatment
T010,A010,Physiotherapy,Standard procedure,1595.67,2023-01-13,Physiotherapy Standard procedure,General Treatment


Databricks visualization. Run in Databricks to view.

In [0]:
# Convert Spark DataFrame to Pandas

pandas_df = df.toPandas()

# Export CSV to local workspace storage

pandas_df.to_csv(
    "/tmp/final_healthcare_data.csv",
    index=False
)

print("Dataset exported successfully to /tmp/")

Dataset exported successfully to /tmp/


In [0]:
display(df)

treatment_id,appointment_id,treatment_type,description,cost,treatment_date,combined_text,detected_features
T001,A001,Chemotherapy,Basic screening,3941.97,2023-08-09,Chemotherapy Basic screening,General Treatment
T002,A002,MRI,Advanced protocol,4158.44,2023-06-09,MRI Advanced protocol,General Treatment
T003,A003,MRI,Standard procedure,3731.55,2023-06-28,MRI Standard procedure,General Treatment
T004,A004,MRI,Basic screening,4799.86,2023-09-01,MRI Basic screening,General Treatment
T005,A005,ECG,Standard procedure,582.05,2023-07-06,ECG Standard procedure,General Treatment
T006,A006,Chemotherapy,Standard procedure,1381.0,2023-06-19,Chemotherapy Standard procedure,General Treatment
T007,A007,Chemotherapy,Advanced protocol,534.03,2023-04-09,Chemotherapy Advanced protocol,General Treatment
T008,A008,Physiotherapy,Basic screening,3413.64,2023-05-24,Physiotherapy Basic screening,General Treatment
T009,A009,Physiotherapy,Standard procedure,4541.14,2023-03-05,Physiotherapy Standard procedure,General Treatment
T010,A010,Physiotherapy,Standard procedure,1595.67,2023-01-13,Physiotherapy Standard procedure,General Treatment
